In [1]:
import sys
import os

project_root = os.path.abspath("../")
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pandas as pd
from data_mining import Dataset
from sklearn.model_selection import train_test_split
from damavand.damavand.signal_processing.transformations import fft
from damavand.damavand.utils import z_score_scaler
from ae import WideConvAutoencoder
from torchinfo import summary
import torch
from utils import plot_error_distribution
import numpy as np
from damavand.damavand.signal_processing.feature_extraction import *

In [3]:
dataset = Dataset(operations=["05"])
data = dataset.mine(win_len=1000, hop_len=1000)

/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP05/bad/M01_Aug_2019_OP05_000.h5  ---  (26793, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP05/bad/M01_Feb_2019_OP05_000.h5  ---  (29831, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP05/bad/M01_Feb_2019_OP05_001.h5  ---  (29621, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP05/bad/M01_Feb_2021_OP05_000.h5  ---  (30000, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP05/good/M01_Aug_2019_OP05_000.h5  ---  (46080, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP05/good/M01_Aug_2019_OP05_001.h5  ---  (41984, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP05/good/M01_Aug_2019_OP05_002.h5  ---  (47104, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/data/M01/OP05/good/M01_Aug_2019_OP05_003.h5  ---  (43008, 3)
/nfs/home/amiber/ml_for_pdm_course_project/CNC_Machining/dat

In [4]:
df = pd.concat(data["OP05"]["0"])
df

,0,1,2,3,4,5,6,7,8,9,...,994,995,996,997,998,999,machine,operation,state,file
0,-9.0,-50.0,-48.0,11.0,17.0,-11.0,-1.0,9.0,19.0,-29.0,...,33.0,37.0,40.0,33.0,19.0,21.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
1,-3.0,27.0,-27.0,-39.0,7.0,9.0,54.0,33.0,23.0,-35.0,...,1188.0,1073.0,1138.0,897.0,1128.0,839.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
2,1007.0,630.0,942.0,595.0,897.0,616.0,650.0,528.0,638.0,452.0,...,439.0,610.0,616.0,757.0,550.0,718.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
3,519.0,573.0,368.0,435.0,203.0,199.0,187.0,54.0,11.0,-97.0,...,95.0,48.0,-39.0,-91.0,-27.0,-58.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
4,35.0,99.0,-17.0,33.0,37.0,3.0,-52.0,-48.0,-33.0,-5.0,...,11.0,1.0,-23.0,15.0,25.0,29.0,M01,OP05,bad,M01_Aug_2019_OP05_000.h5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,-7.0,-25.0,13.0,-37.0,5.0,-11.0,-5.0,-21.0,0.0,-15.0,...,-15.0,-17.0,-13.0,-15.0,-9.0,-11.0,M01,OP05,good,M01_Feb_2021_OP05_008.h5
43,-15.0,-17.0,-7.0,-13.0,-13.0,-21.0,-11.0,-3.0,-5.0,-15.0,...,-11.0,-5.0,-37.0,-7.0,-27.0,-7.0,M01,OP05,good,M01_Feb_2021_OP05_008.h5
44,-7.0,-31.0,-13.0,-33.0,-17.0,-31.0,-3.0,-44.0,-7.0,-21.0,...,-21.0,-13.0,-11.0,-21.0,-17.0,-15.0,M01,OP05,good,M01_Feb_2021_OP05_008.h5
45,-17.0,-7.0,-17.0,-5.0,-15.0,-5.0,-15.0,-15.0,-19.0,-15.0,...,-13.0,-5.0,-21.0,-19.0,0.0,-29.0,M01,OP05,good,M01_Feb_2021_OP05_008.h5


In [5]:
df_train, df_test = train_test_split(df, test_size=0.4)
x_train, y_train = df_train.iloc[:, : -4], df_train.iloc[:, -4 :]
x_test, y_test = df_test.iloc[:, :-4], df_test.iloc[:, -4:]

In [17]:
import scipy

time_features = {
  'mean': (np.mean, (), {}),
  'std': (np.std, (), {}),
  'smsa': (smsa, (), {}),
  'rms': (rms, (), {}),
  'peak': (peak, (), {}),
  'skew': (scipy.stats.skew, (), {}),
  'kurtosis': (scipy.stats.kurtosis, (), {}),
  'crest_factor': (crest_factor, (), {}),
  'clearance_factor': (clearance_factor, (), {}),
  'shape_factor': (shape_factor, (), {}),
  'impulse_factor': (impulse_factor, (), {}),
}
x_train_features = feature_extractor(x_train, time_features)
x_test_features = feature_extractor(x_test, time_features)

In [18]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

x_train_features_sclaed = scaler.fit_transform(x_train_features)
x_test_features_sclaed = scaler.transform(x_test_features)

In [19]:
from sklearn.svm import OneClassSVM

model = OneClassSVM()
model.fit(x_train_features_sclaed)


y_pred_train = model.predict(x_train_features_sclaed)
y_pred_test = model.predict(x_test_features_sclaed)

In [20]:
flags = np.where(y_pred_test == -1, "bad", "good").tolist()

In [21]:
y_test.shape

(622, 4)

In [22]:
y_pred_test.shape

(622,)

In [23]:
from sklearn.metrics import classification_report

print(classification_report(y_test["state"], flags))

              precision    recall  f1-score   support

         bad       0.08      0.49      0.13        51
        good       0.91      0.47      0.62       571

    accuracy                           0.47       622
   macro avg       0.49      0.48      0.38       622
weighted avg       0.84      0.47      0.58       622



In [24]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test["state"], flags)

array([[ 25,  26],
       [303, 268]])